In [1]:
# Importing Necessary Libraries

import numbers
import os
import sys
import matplotlib.collections
import matplotlib.pyplot
import argparse
import numpy

In [2]:
# Trying to understand the first argument given on the file:

matplotlib.use('AGG')

#### Explanation ####
# In Matplotlib, an AGG filter, or Anti-Grain Geometry filter, is used to apply
# certain transformations to graphical elements, such as lines, markers, or
# text, before they are displayed on a plot. The AGG filter allows us to modify
# the appearance of elements in a plot, such as adjusting their transparency,
# blurring them, or applying other visual effects.

In [3]:
def parse_args():
    """
        Parses inputs from the commandline.
        :return: inputs as a Namespace object
    """
    parser = argparse.ArgumentParser(description="Draws and saves a ROC plot for one of OR all the three "
                                                 "impact predictors (SIFT, PolyPhen, and BLOSUM62) to a file")

    # Arguments
    parser.add_argument("-ipred", "--input_predictor", help="tab-separated file with predictor scores. "
                                                            "This argument is required!",
                        action='append', required=True)
    parser.add_argument("-ibench", "--input_benchmark", help="tab-separated benchmark classification file. "
                                                             "This argument is required!", required=True)
    parser.add_argument("-color", "--use_color_roc_plot", help="plot ROC with gradient color", action
    ='store_true', required=False)
    parser.add_argument("-o", "--out_filepath", help="a path to write the output .png file with a ROC plot. "
                                                     "This argument is required!", required=True)

    return parser.parse_args()

In [4]:
def parse_predictor(filename):
    """
        Parses scores of every HGVS ID out of the predictor input file.
        :param filename: a str with the predictor input file
        :return: a dict with HGVS IDs (keys), and the corresponding predictor scores (values)
    """

    global type_predictor
    if 'sift' in filename:
        type_predictor = 'sift'
    elif 'polyphen' in filename:
        type_predictor = 'polyphen'
    elif 'baseline' in filename:
        type_predictor = 'BLOSUM'
    else:
        type_predictor = ''

    predictor_dict = {}

    with open(filename, 'r') as f:
        # Total bytes in the file (end of file)
        eof = f.seek(0, 2)
        # Go to the beginning of the file again
        f.seek(0)
        # Read the first line (should be the header)
        f.readline()
        # Get the current position of the file pointer
        cur = f.tell()
        # If the file doesn't contain predictor results (the header excluded), exit
        if cur == eof:
            sys.exit('ERROR: input predictor file does not contain predictor results!')

        # To check what arr holds:
        arr_check = []

        for line in f:
            line = line.rstrip()
            arr_step = line.split("\t")

            if len(arr_step) != 2:
                print("Warning: the following line does not have two elements separated by a tab:\n", line)

            key = (arr_step[0])
            value = float(arr_step[1])
            arr_check = arr_step # We will store a specific arr_step's value in the var_check
            predictor_dict[key] = value

    f.close()
    return predictor_dict, arr_check

    ## Explanation: This code is just for extracting HGVS_IDs and their scores of SIFT and PolyPhen Results

In [5]:
###### Seing the structure of the arr list and predictor_score ######

## First call the function and hold the outcome in a variable
predictor_dict, arr_check = parse_predictor("/Users/yigitatagulsener/Desktop/Masters/Introduction_to_Bioinformatics/FoB/Data/vep/HGVS_2020_big_sift_scores.tsv")

# For predictor_dict structure:
print("The predictor_dict structure is (by using predictor_dict.keys() and predictor_dict.values()):\n\n",predictor_dict.keys(),"\n\n",predictor_dict.values())

The predictor_dict structure is (by using predictor_dict.keys() and predictor_dict.values()):

 dict_keys(['NC_000005.10:g.43702652T>G', 'NC_000008.11:g.24394146T>C', 'NC_000014.9:g.35007370A>G', 'NC_000023.11:g.48523912G>T', 'NC_000007.14:g.92086295A>G', 'NC_000005.10:g.131159578A>G', 'NC_000017.11:g.38335201G>A', 'NC_000010.11:g.97601884C>A', 'NC_000002.12:g.176119206T>C', 'NC_000023.11:g.106037037C>A', 'NC_000002.12:g.31529385G>T', 'NC_000009.12:g.135778780T>G', 'NC_000002.12:g.55683831C>T', 'NC_000023.11:g.150598600G>A', 'NC_000015.10:g.40611486A>G', 'NC_000002.12:g.99361611A>T', 'NC_000021.9:g.46353263C>T', 'NC_000023.11:g.139561992C>A', 'NC_000002.12:g.20003167A>T', 'NC_000003.12:g.181712650T>C', 'NC_000019.10:g.7120739C>T', 'NC_000011.10:g.36574635C>T', 'NC_000001.11:g.167127158C>T', 'NC_000002.12:g.165991510A>C', 'NC_000009.12:g.119167448G>A', 'NC_000016.10:g.75530369G>A', 'NC_000003.12:g.31533134G>A', 'NC_000023.11:g.139537369T>G', 'NC_000016.10:g.177340C>T', 'NC_000016.10:g.1

In [6]:
# For arr_check structure:
print("The arr_check structure is:\n",arr_check)

# So what is the value of arr_step in this case? (We used arr_check to store arr_step)
print("\nThe first element of arr_step is the ID:\n",arr_check[0],"\n\nThe second element of arr_step is the value:\n",arr_check[1])

The arr_check structure is:
 ['NC_000008.11:g.142913391T>C', '0.01']

The first element of arr_step is the ID:
 NC_000008.11:g.142913391T>C 

The second element of arr_step is the value:
 0.01


In [7]:
def parse_benchmark(filename):
    """
        Parses every HGVS classification out of the benchmark file.
        :param filename: a str with the benchmark input file
        :return: a dict with HGVS IDs (keys), and corresponding benchmark classifications (values)
    """

    benchmark_dict = {}

    with open(filename,'r') as f:
        # Total bytes in the file (end of file)
        eof = f.seek(0, 2)
        # Go to the beginning of the file again
        f.seek(0)
        # Read the first line (should be the header)
        f.readline()
        # Get the current position of the file pointer
        cur = f.tell()
        # If the file doesn't contain predictor results (the header excluded), exit
        if cur == eof:
            sys.exit('ERROR: input benchmark file does not contain benchmark results!')

            arr_check = []

        for line in f:
            line = line.rstrip()
            arr_step = line.split("\t")

            if len(arr_step) < 2:
                print("Warning: the following line does not have three elements separated by a tab:\n", line)

            arr_check = arr_step
            key = arr_step[0]
            value = arr_step[1]
            benchmark_dict[key] = value

    return benchmark_dict, arr_check

    ## Explanation: This code is just for extracting HGVS_IDs and their scores from the benchmark

In [8]:
#### To understand the structure of parse_benchmark function ####

## Calling the function and assigning the outcome to the variables
benchmark_dict, arr_check = parse_benchmark("/Users/yigitatagulsener/Desktop/Masters/Introduction_to_Bioinformatics/FoB/Data/HGVS_2020_big_benchmark.tsv")
# For the structure of benchmark_dict
print("The keys of benchmark_dict:\n",benchmark_dict.keys(),"\n\nThe values of benchmark_dict:\n", benchmark_dict.values())

The keys of benchmark_dict:
 dict_keys(['NC_000016.10:g.89748658C>T', 'NC_000008.11:g.143818378G>A', 'NC_000023.11:g.32644238T>A', 'NC_000002.12:g.39022779A>G', 'NC_000001.11:g.154926384G>A', 'NC_000001.11:g.215674313G>C', 'NC_000002.12:g.112756449T>C', 'NC_000012.12:g.32643663G>A', 'NC_000001.11:g.215675336C>T', 'NC_000002.12:g.159882292G>A', 'NC_000003.12:g.184165474C>T', 'NC_000001.11:g.92475800A>G', 'NC_000014.9:g.91969063T>G', 'NC_000012.12:g.49185198G>A', 'NC_000009.12:g.95467285G>T', 'NC_000012.12:g.49034915C>A', 'NC_000001.11:g.11012634G>A', 'NC_000016.10:g.8768220C>T', 'NC_000015.10:g.92998538C>T', 'NC_000007.14:g.4788228G>A', 'NC_000018.10:g.62095902C>T', 'NC_000005.10:g.122074012C>T', 'NC_000002.12:g.25244154C>G', 'NC_000017.11:g.42322445G>C', 'NC_000003.12:g.4361853C>T', 'NC_000009.12:g.136674753T>A', 'NC_000016.10:g.74628505A>T', 'NC_000003.12:g.190388374C>G', 'NC_000009.12:g.126693528G>C', 'NC_000011.10:g.47410315G>A', 'NC_000019.10:g.17766667A>C', 'NC_000023.11:g.3836739

In [9]:
# For the structure of arr_check
print("The structure of arr_check is:\n",arr_check,"\n\nThe first element is ID:\n",arr_check[0],"\n\nThe second element is the tag (pathogenic/benign):\n",arr_check[1])

The structure of arr_check is:
 ['NC_000017.11:g.18122113G>T', 'Pathogenic'] 

The first element is ID:
 NC_000017.11:g.18122113G>T 

The second element is the tag (pathogenic/benign):
 Pathogenic


In [10]:
def count_total_results(predictor_score_dict, benchmark_dict):
    """
        Calculates the total number of positives (P), or pathogenic results, and negatives (N), or benign results.
        :param predictor_score_dict: a dict of all predictor scores
        :param benchmark_dict: a dict of benchmark classifications
        :return: a list of ints for the total number of pathogenic and benign results
    """

    pathogenic = 0
    benign = 0
    for key, value in predictor_score_dict.items():
        result = benchmark_dict[key]
        if result == 'Pathogenic':
            pathogenic += 1
        elif result == 'Benign':
            benign += 1
    return [pathogenic, benign]

    ## Explanation: This code takes the HGVS_ID from the prediction model dictionary that we created above and uses the ID as a key to search
    # in the benchmark_dict (which is the ClinVar dataset. At the end, total amount of pathogenic and benign results from prediction model will
    # be observed

In [11]:
#### Understand the structure of count_total_results() function ####

## Call the fucntion and assign the outcome to variables
[pathogenic, benign] = count_total_results(predictor_dict, benchmark_dict)

# Printing the pathogenic and benign numbers
print("\nThe number of pathogenic tag is:\n",pathogenic,"\n\nThe number of benign tag is:\n",benign,"\n")


The number of pathogenic tag is:
 600 

The number of benign tag is:
 600 



In [14]:
def calculate_coordinates(predictor_score_dict, benchmark_dict, out_filepath):
    """
        Calculates coordinates of x and y based on the predictor scores.
        :param predictor_score_dict: a dictionary with scores produced by parse_predictor()
        :param benchmark_dict: a dictionary with benchmark classifications produced by parse_benchmark()
        :param out_filepath: a str with the output .png file path
        :return: lists of coordinates for the ROC plot (TPR and FPR), and a list of sorted predictor scores
    """

    # Get a list of tuples from predictor_score_dict: (predictor score, HGVS ID)
    # Normally when we look at the structure of predictor_score_dict. It can be seen that it is IDs followed by scores. But in this function
    # we use sorted function and in order to use this function we should have scores followed by IDs. Therefore we do the code below:
    score_hgvs_pairs = [(v, k) for k, v in predictor_score_dict.items()]

    sorted_score_hgvs_pairs = score_hgvs_pairs
    
    #########################
    ### START CODING HERE ###
    #########################
    # You need to sort the scores in the correct order for the ROC plot.
    # Use the following if-statement and replace the question mark with the type of the predictor.
    # It will put the ROC curve at the correct side of the diagonal line.

    # if type_predictor == ? :
    #     sorted_score_hgvs_pairs = sorted(score_hgvs_pairs)
    # else:
    #     sorted_score_hgvs_pairs = sorted(score_hgvs_pairs, reverse=True)

    ## the sorted() function sort the values from smaller to biggest
    # So in SIFT scores: 0.0 is deletirous and 1.0 is benign, it is kind of samo in BLOSSUM62, negative means deletirous and positive
    # means natural / benign so we should use sorted() function as it is
    # But when we look at the PolyPhen scores it is the opposite. The 1.0 means deletirous and 0.0 means benign. So we add "reverse = TRUE"
    # part to reverse order the sorted() function
    
    if type_predictor == 'sift' or type_predictor == 'BLOSUM' :
        sorted_score_hgvs_pairs = sorted(score_hgvs_pairs) # 0 means pathogenic, 1 means benign
    else:
        sorted_score_hgvs_pairs = sorted(score_hgvs_pairs, reverse=True)

    #########################
    ###  END CODING HERE  ###
    #########################

    # Later, each coordinate in the ROC plot will be associated with a predictor score (a threshold score). Thus, we
    # need a separate list for predictor scores
    coordinate_score = [sorted_score_hgvs_pairs[0][0]]

    # Create lists to store coordinates (tpr, fpr). Starts in (0,0)
    tpr = [0.0]
    fpr = [0.0]

    # Create variables to keep track of the number of true positives (TPs) and false positives (FPs) [As we are going to do ROC curves]
    num_tp = 0
    num_fp = 0

    # Get the total number of positives (P) and negatives (N)
    # total_p = pathogenic number, total_n = benign number (600 to 600)
    total_p, total_n = count_total_results(predictor_score_dict, benchmark_dict)

    # Get a list of indices of scores before breakpoints
    # A breakpoint is the place in the sorted scores list where the score changes.
    # the index_prebreakpoint_score list will hold the indices just before the change. 
    # this break point is going to act like different threshold values for our ROC curve
    # In our sorted scores list there would be values which are the same (for example: there can be value 0.1 three times)
    # in this case we just need to store the number 0.1 for the threshold. I tried to explain what each line does.
    
    index_prebreakpoint_score = [] # list that we are going to store breakpoints' index. It is empty when we start
    previous_score = sorted_score_hgvs_pairs[0][0] # In order to make compare of the values in the loop we give the first value to be compared
                                                   # before the for loop
    for i in range(len(sorted_score_hgvs_pairs)): # In order to have i as an integer number we used range(len()).
        score = sorted_score_hgvs_pairs[i][0] # You take the i'th indexed value
                                              # As it was explained before the structure of this list is value followed by ID so we use [0]
        if previous_score != score: # If the previous value is not equal to the current value this means change so we should get the index
            # Add index of the score before the breakpoint
            index_prebreakpoint_score.append(i - 1)
        previous_score = score

    # Add index of the last score (for the last coordinate)
    index_prebreakpoint_score.append(len(sorted_score_hgvs_pairs) - 1) # In the loop we compare the previous value to the current value
                                                                       # but when we reach the last element of the list there is no value 
                                                                       # to compare so with this line you manually assign the last element
                                                                       # as a index

    # Iterate over HGVS IDs of SNPs and corresponding sorted predictor scores
    for i in range(len(sorted_score_hgvs_pairs)):
        score = sorted_score_hgvs_pairs[i][0]
        hgvs = sorted_score_hgvs_pairs[i][1]

        #########################
        ### START CODING HERE ###
        #########################
        # Determine whether the SNP is classified by the benchmark as:
        #    Pathogenic -> actual positive, thus a true positive (y-coordinate)
        #    Benign     -> actual negative, thus a false positive (x-coordinate)
        
        # The logic behind this true positive and false positive:
        # So the hgvs ID that you are using here is from sorted predicter score. I will clarify the logic using an example:
        # lets think that our first predictor is SIFT. In SIFT 0 is deterious and 1 is benign. In the code above we sorted the SIFT scores
        # So in this loop we start from 0 and than increment. You take the ID of the 0'th indexed element and search it up on the benchmark_dict
        # You expect it to be pathogenic. If it is pathogenic it is true positive and if the result is benign it is a false positive

        # Increase the respective value of num_fp or num_tp
        # In each run you are increasing the true positive and true negative value

        result = benchmark_dict[hgvs]
        if result == 'Pathogenic':
            num_tp += 1
        elif result == 'Benign':
            num_fp += 1

        # Now, you need to calculate TPR and FPR for unique scores as TP/P and FP/N, respectively,
        # using num_fp, num_tp, total_n, and total_p correctly. Append the values
        # to the corresponding lists: tpr is a list of y-coordinates and fpr is a list of x-coordinates.
        # Calculate the rates if HGVS score index i is the index of the score before a breakpoint
        # (use index_prebreakpoint_score). Also, append the score to coordinate_score.
        # Note: you are calculating true positive rate and false positive rate just in the break points (threshold values)

        if i in index_prebreakpoint_score:
            tpr.append(num_tp / total_p)
            fpr.append(num_fp / total_n)
            coordinate_score.append(score)

        #########################
        ###  END CODING HERE  ###
        #########################
    if out_filepath:
        out_dir, out_filename = os.path.split(out_filepath)
        # Write coordinates to a .tsv file
        with open(os.path.join(out_dir, out_filename.split('.')[0] + '_xy.tsv'), 'w') as f:
            for a, b in zip(fpr, tpr):
                f.write(str(a) + '\t' + str(b) + '\n')

    return tpr, fpr, coordinate_score

In [15]:
## Understanding the coordinate_score structure
tpr, fpr, coordinate_score = calculate_coordinates(predictor_dict, benchmark_dict,"/Users/yigitatagulsener/Desktop/Masters/Introduction_to_Bioinformatics/FoB/Output/Coordinates.txt")
print("\n\nThe coordinate scores are:\n",coordinate_score)



The coordinate scores are:
 [0.0, 0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2, 0.21, 0.22, 0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.29, 0.3, 0.31, 0.32, 0.33, 0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.4, 0.41, 0.42, 0.43, 0.44, 0.45, 0.46, 0.47, 0.48, 0.49, 0.5, 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57, 0.58, 0.59, 0.6, 0.62, 0.63, 0.64, 0.65, 0.66, 0.67, 0.69, 0.7, 0.73, 0.74, 0.75, 0.76, 0.77, 0.78, 0.79, 0.81, 0.82, 0.84, 0.85, 0.86, 0.87, 0.9, 0.91, 0.92, 0.93, 0.95, 0.96, 0.97, 1.0]


In [16]:
def integrate(fpr, tpr):
    """
        Calculates the Area Under the Curve (AUC) for a given list of coordinates.
        :param fpr: a list of FPRs
        :param tpr: a list of TPRs
        :return: a float with AUC
    """

    auc = 0.
    last_fpr = fpr[0]
    last_tpr = tpr[0]

    for cur_fpr, cur_tpr in list(zip(fpr, tpr))[1:]:
        #########################
        ### START CODING HERE ###
        #########################
        # Calculate AUC
        #This is a well knowed formula so ı just implamented it here (for more information you can search for: Linear trapezoidal method)
        
        width = cur_fpr - last_fpr
        tp_average = (last_tpr + cur_tpr) / 2
        auc += width * avg_height

        #########################
        ###  END CODING HERE  ###
        #########################
        last_fpr = cur_fpr
        last_tpr = cur_tpr

    return auc

In [22]:
def roc_plot(tpr, fpr, coordinator_score, out_filepath, color = False):
    """
       Draws ROC plot with gradient color.
       :param tpr: a list of TPRs
       :param fpr: a list of FPRs
       :param coordinator_score: a list of predictor scores
       :param out_filepath: a str with the output .png file path
       :param color: boolean (False by default) to enable gradient color plotting
    """

    # Compute AUC
    auc = integrate(fpr, tpr)

    # Draw ROC plot and write it to a file
    lw = 1
    figure, axes = matplotlib.pyplot.subplots(1, 1)

    if color:
        lc = colorline(fpr, tpr, coordinator_score, axes=axes)
        color_bar = figure.colorbar(lc)
        colorbar_legend = type_predictor + ' score'
        color_bar.ax.set_ylabel(colorbar_legend)
    else:
        axes.plot(fpr, tpr)

    axes.plot((0, 1), (0, 1), '--', color='navy', lw=lw, linestyle='--', label='Random')
    axes.set_xlim([-0.008, 1.008])
    axes.set_ylim([-0.008, 1.008])
    axes.set_xlabel('False Positive Rate')
    axes.set_ylabel('True Positive Rate')
    axes.set_title('AUC = %.3f' % auc)
    matplotlib.pyplot.savefig(out_filepath)

def roc_plot_together(list_tpr, list_fpr, labels, out_filepath):
    """
       Draws ROC plot for three predictors in one figure without gradient color.
       :param list_tpr: a list of lists with TPRs for each predictor
       :param list_fpr: a list of lists with FPRs for each predictor
       :param labels: a list with labels for each of the three ROC curves
       :param out_filepath: a str with output .png file path
    """

    lw = 1
    list_color = ['g','r','m']
    figure, axes = matplotlib.pyplot.subplots(1, 1)

    for tpr, fpr, color, label in zip(list_tpr, list_fpr, list_color, labels):
        auc = integrate(fpr, tpr)
        line_label = '{} (AUC= {:.3f})'.format(label, auc)
        axes.plot(fpr, tpr, c=color, label=line_label)

    axes.plot((0, 1), (0, 1), '--', color='navy', lw=lw, linestyle='--', label='Random')
    axes.set_xlim([-0.008, 1.008])
    axes.set_ylim([-0.008, 1.008])
    axes.legend()
    axes.set_xlabel('False Positive Rate')
    axes.set_ylabel('True Positive Rate')
    matplotlib.pyplot.savefig(out_filepath)

def colorline(x, y, z=None, axes=None, cmap=matplotlib.pyplot.get_cmap('coolwarm'), linewidth=3, alpha=1.0, **kwargs):
    """
        Plots a colored line with coordinates x and y. Optionally, specify colors in the array z. Optionally,
        specify a colormap, a norm function and a line width.
        :param x: a list of x-coordinates
        :param y: a list of y-coordinates
    """

    def make_segments(x, y):
        """
            Creates a list of line segments from x- and y-coordinates in the correct format for LineCollection:
            an array of the form numlines x (points per line) x 2 (x and y) array.
            :param x: a list of x-coordinates
            :param y: a list of y-coordinates
            :return: a list of line segments
        """

        points = numpy.array([x, y]).T.reshape(-1, 1, 2)
        segments = numpy.concatenate([points[:-1], points[1:]], axis=1)

        return segments

    # Default colors equally spaced on [0,1]:
    if z is None:
        z = numpy.linspace(0.0, 1.0, len(x))

    # Special case if a single number:
    if isinstance(z, numbers.Real):
        z = numpy.array([z])

    z = numpy.asarray(z)

    segments = make_segments(x, y)
    lc = matplotlib.collections.LineCollection(segments, array=z, cmap=cmap, linewidth=linewidth,
                                               alpha=alpha, **kwargs)

    if axes is None:
        axes = matplotlib.pyplot.gca()

    axes.add_collection(lc)
    axes.autoscale()

    return lc



In [23]:
def main():

    # Process arguments
    args = parse_args()
    predictor_path = args.input_predictor
    benchmark_path = args.input_benchmark
    out_filepath = args.out_filepath
    color = args.use_color_roc_plot

    out_dir, out_filename = os.path.split(out_filepath)
    # Check if output filename contains .png extension
    if '.png' not in out_filename:
        sys.exit(r'ERROR: filename "%s" in the output file path argument should contain .png extension!' % out_filename)

    # Check if output directory exists
    if not os.path.exists(out_dir):
        sys.exit(r'ERROR: output directory "%s" to store the ROC plot does not exist! Follow instructions in'
                 r' the manual!' % out_dir)

    # Parse input files, calculate ROC coordinates and plot
    if len(predictor_path) == 1:
        predictor_path = predictor_path[0]
        # Parse predictor and benchmark files
        predictor_results = parse_predictor(predictor_path)
        benchmark_results = parse_benchmark(benchmark_path)
        # Calculate ROC coordinates
        tpr, fpr, coordinate_score = calculate_coordinates(predictor_results, benchmark_results, out_filepath)
        # Draw and save the ROC plot
        roc_plot(tpr, fpr, coordinate_score, out_filepath, color)

    elif len(predictor_path) != 3:
        sys.exit('ERROR: to plot three predictors (baseline, sift, and polyphen) all together, '
              'please input three files by adding -ipred before each file')

    else:
        # Lists to store labels and coordinates for three predictors
        labels = []
        list_tpr = []
        list_fpr = []
        # For each predictor
        for predictor in predictor_path:
            # Parse predictor and benchmark files
            predictor_results = parse_predictor(predictor)
            benchmark_results = parse_benchmark(benchmark_path)
            # Append predictor type to the label list
            labels.append(type_predictor)
            # Calculate ROC coordinates
            tpr, fpr, coordinate_score = calculate_coordinates(predictor_results, benchmark_results, None)
            # Append coordinates to lists
            list_tpr.append(tpr)
            list_fpr.append(fpr)
        # Plot ROC curves for three predictors together
        roc_plot_together(list_tpr, list_fpr, labels, out_filepath)


if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] -ipred INPUT_PREDICTOR -ibench
                             INPUT_BENCHMARK [-color] -o OUT_FILEPATH
ipykernel_launcher.py: error: the following arguments are required: -ipred/--input_predictor, -ibench/--input_benchmark, -o/--out_filepath


SystemExit: 2